# mms-300m karaoke finetune

In [ ]:
import json
import re
import unicodedata
from typing import Any, cast

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import login as hf_login
from transformers import (
    EarlyStoppingCallback,
    EvalPrediction,
    Trainer,
    TrainingArguments,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    set_seed,
)
from transformers.modeling_outputs import CausalLMOutput

In [ ]:
MODEL_BASE = "facebook/mms-300m"
MODEL_OUTPUT_DIR = "mms-300m-ForcedAligner-karaoke-ja-Latn"
FINGERPRINT_PREFIX = "karaoke-fa"
MAX_DURATION_SECONDS = 150

set_seed(42)

In [ ]:
hf_login()

In [ ]:
dataset = load_dataset("NextFire/karaoke-alignments", split="train")

dataset = dataset.filter(
    lambda x: x.metadata.duration_seconds <= MAX_DURATION_SECONDS,
    input_columns=["audio"],
    new_fingerprint=f"{FINGERPRINT_PREFIX}_filtered",
)


def normalize_morae(morae):
    for mora in morae:
        value = cast(str, mora["value"])
        value = value.casefold()
        value = value.replace("’", "'")
        value = unicodedata.normalize("NFKD", value)
        value = value.encode("ascii", "ignore").decode("ascii")
        value = re.sub(r"[^a-z']", " ", value)
        value = re.sub(r" +", " ", value)
        mora["value"] = value
    return {"morae": morae}


dataset = dataset.map(
    normalize_morae,
    input_columns=["morae"],
    writer_batch_size=500,
    new_fingerprint=f"{FINGERPRINT_PREFIX}_normalized",
)

In [ ]:
AUTO_SPLIT_RE = re.compile(
    r"(?i)(?:(?<=[^sc])(?=h))|(?:(?<=[^kstnhfmrwpbdgzcj])(?=y))|(?:(?<=[^t])(?=s))|(?:(?=[ktnfmrwpbdgzcjlqvx]))|(?:(?<=[aeiou]|\W)(?=[aeiou]))"
)


def auto_split(word: str):
    splitter_str, _ = AUTO_SPLIT_RE.subn("#@", word)
    syllables = re.split(r"#@| |'", splitter_str, flags=re.MULTILINE)
    syllables = list(filter(None, syllables))
    return syllables


vocab = dataset.map(
    lambda batch: {
        "vocab": list(
            set(
                split
                for sample in batch
                for mora in sample
                for split in auto_split(mora["value"])
            )
        )
    },
    input_columns=["morae"],
    remove_columns=dataset.column_names,
    batched=True,
    batch_size=None,
    keep_in_memory=True,
)["vocab"]

vocab_dict = {
    "|": 0,
    "[UNK]": 1,
    "[PAD]": 2,
    "'": 3,
    **{v: k for k, v in enumerate(sorted(vocab), start=4)},
}

with open("vocab.json", "w") as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=2)

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    word_delimiter_token="|",
    unk_token="[UNK]",
    pad_token="[PAD]",
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained(MODEL_OUTPUT_DIR)

In [ ]:
class Wav2Vec2ForForcedAligner(Wav2Vec2ForCTC):
    def forward(
        self,
        input_values: torch.Tensor | None,
        attention_mask: torch.Tensor | None = None,
        output_attentions: bool | None = None,
        output_hidden_states: bool | None = None,
        return_dict: bool | None = None,
        labels: torch.Tensor | None = None,
        **kwargs,
    ):
        outputs = super().forward(
            input_values=input_values,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            labels=None,
            **kwargs,
        )
        assert isinstance(outputs, CausalLMOutput)
        assert outputs.logits is not None
        assert labels is not None
        loss = F.cross_entropy(outputs.logits.transpose(1, 2), labels)
        return CausalLMOutput(
            loss=loss,  # pyright: ignore[reportArgumentType]
            logits=outputs.logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


model = Wav2Vec2ForForcedAligner.from_pretrained(
    MODEL_BASE,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
    pad_token_id=tokenizer.pad_token_id,
    vocab_size=len(tokenizer),
)

# model.freeze_feature_encoder()

In [ ]:
def prepare_dataset(example: dict[str, Any]):
    audio = example["audio"]
    input_values = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],  # pyright: ignore[reportCallIssue]
    ).input_values[0]
    example["input_values"] = input_values

    output_length = int(model._get_feat_extract_output_lengths(len(input_values)))
    duration_ms = float(audio.metadata.duration_seconds * 1000)
    frames_per_ms = output_length / duration_ms
    labels = [tokenizer.pad_token_id] * output_length
    boundaries = [0] * output_length
    for mora in example["morae"]:
        value = cast(str, mora["value"])
        if not value.strip():
            continue
        start_frame = round(mora["start"] * frames_per_ms)
        assert start_frame < output_length
        end_frame = round(mora["end"] * frames_per_ms)
        assert start_frame <= end_frame <= output_length
        n_frames = end_frame - start_frame
        input_ids = processor(text=mora["value"]).input_ids
        for idx, frame in enumerate(range(start_frame, end_frame)):
            if labels[frame] == tokenizer.pad_token_id:
                token_idx = (idx * len(input_ids)) // n_frames
                labels[frame] = input_ids[token_idx]
            else:
                # Mask overlapping alignments with -100 to ignore them in loss calculation
                labels[frame] = -100
        boundaries[start_frame] = 1
        if end_frame < output_length:
            boundaries[end_frame] = 1
    example["labels"] = labels
    example["boundaries"] = boundaries

    return example


dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names,
    num_proc=8,
    new_fingerprint=f"{FINGERPRINT_PREFIX}_prepared",
)

In [ ]:
def data_collator(examples: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
    input_features = [{"input_values": e["input_values"]} for e in examples]
    labels = [{"input_ids": e["labels"]} for e in examples]
    boundaries = [{"input_ids": e["boundaries"]} for e in examples]
    batch = processor.pad(input_features, padding=True, return_tensors="pt")
    l_batch = processor.pad(labels=labels, padding=True, return_tensors="pt")
    b_batch = processor.pad(labels=boundaries, padding=True, return_tensors="pt")
    batch["labels"] = l_batch["input_ids"].masked_fill(
        l_batch.attention_mask.ne(1), -100
    )
    batch["boundaries"] = b_batch["input_ids"].masked_fill(
        b_batch.attention_mask.ne(1), -100
    )
    return batch


def compute_metrics(eval_pred: EvalPrediction):
    logits = eval_pred.predictions
    labels, boundaries = eval_pred.label_ids  # pyright: ignore[reportAssignmentType]
    predictions = np.argmax(logits, axis=-1)

    TOLERANCE = 2  # frames (~40ms at ~20ms/frame)

    precision_list: list[float] = []
    recall_list: list[float] = []
    for pred_i, label_i, bound_i in zip(predictions, labels, boundaries):
        valid = label_i != -100
        pred_i = pred_i[valid]
        bound_i = bound_i[valid]

        true_bounds = np.where(bound_i == 1)[0]
        # All token transitions: catches both PAD<->non-PAD and adjacent mora
        # boundaries (back-to-back moras with no silence gap).
        # Intra-mora character cycling also fires here (lowers precision) but
        # recall over true mora boundaries remains complete.
        pred_bounds = np.where(np.diff(pred_i) != 0)[0] + 1

        if len(true_bounds) == 0 and len(pred_bounds) == 0:
            continue
        elif len(true_bounds) == 0 or len(pred_bounds) == 0:
            precision_list.append(0.0)
            recall_list.append(0.0)
        else:
            dist = np.abs(pred_bounds[:, None] - true_bounds[None, :])
            precision_list.append(float(np.any(dist <= TOLERANCE, axis=1).mean()))
            recall_list.append(float(np.any(dist <= TOLERANCE, axis=0).mean()))

    mean_p = float(np.mean(precision_list)) if precision_list else 0.0
    mean_r = float(np.mean(recall_list)) if recall_list else 0.0
    f1 = 2 * mean_p * mean_r / (mean_p + mean_r) if (mean_p + mean_r) > 0 else 0.0

    return {
        "boundary_precision": mean_p,
        "boundary_recall": mean_r,
        "boundary_f1": f1,
    }

In [ ]:
split = dataset.train_test_split(test_size=0.1)

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=4,
    eval_accumulation_steps=1,
    gradient_checkpointing=True,
    bf16=True,
    tf32=True,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    label_names=["labels", "boundaries"],
    metric_for_best_model="boundary_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    logging_steps=25,
    report_to="trackio",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [ ]:
trainer.train()
trainer.save_model()